# Notebook 08 — Signal Candidate Mapping (SOC Detection)

## Phase 3.5 (CICIDS) — SOC Detection Validation Layer

### Objective
Convert SOC detection ideas from the **CyberOps SOC design phase** into **feasible network telemetry detections** using CICIDS-2017 data.

This notebook answers:

1. **Which SOC alerts are feasible** from CICIDS flow telemetry?
2. **Which CICIDS features** map to each alert category?
3. What is the **expected false-positive risk** for each alert?

---

## Why this notebook matters
CICIDS is not a “logins dataset”.
It is a **network flow dataset**, so detections must be designed accordingly.

This notebook formalizes:

**SOC detection idea → Feasible signal in CICIDS → Candidate columns → Known risks**

---

## Outputs
This notebook exports:

1. `outputs/tables/07_signal_candidate_mapping.csv`
2. `outputs/tables/07_feature_availability.csv`

These are “project artifacts” and should remain stable.


In [3]:
import numpy as np
import pandas as pd
from pathlib import Path


In [18]:
import sys
from pathlib import Path

# Primary: your local path (fast)
PHASE_ROOT = Path(r"D:\Projects\CICIDS-2017\03-security-integration\cicids-2017\phase-3.5-soc-detection").resolve()

# Fallback: find phase folder from current working directory
if not PHASE_ROOT.exists():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        candidate = p / "03-security-integration" / "cicids-2017" / "phase-3.5-soc-detection"
        if candidate.exists():
            PHASE_ROOT = candidate.resolve()
            break

if not PHASE_ROOT.exists():
    raise RuntimeError("Could not locate phase-3.5-soc-detection folder. Fix PHASE_ROOT path.")

if str(PHASE_ROOT) not in sys.path:
    sys.path.insert(0, str(PHASE_ROOT))

print("PHASE_ROOT:", PHASE_ROOT)


PHASE_ROOT: D:\Projects\CICIDS-2017\03-security-integration\cicids-2017\phase-3.5-soc-detection


In [5]:
OUTPUT_DIR = PHASE_ROOT / "outputs"
OUTPUT_TABLES = OUTPUT_DIR / "tables"
OUTPUT_FIGS = OUTPUT_DIR / "figures"

OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGS.mkdir(parents=True, exist_ok=True)

OUTPUT_TABLES, OUTPUT_FIGS


(WindowsPath('D:/Projects/CICIDS-2017/03-security-integration/cicids-2017/phase-3.5-soc-detection/outputs/tables'),
 WindowsPath('D:/Projects/CICIDS-2017/03-security-integration/cicids-2017/phase-3.5-soc-detection/outputs/figures'))

In [6]:
from utils.cicids_loader import load_cicids_multi


In [7]:
FILES_FOR_PHASE_3_5 = [
    "Monday-WorkingHours.pcap_ISCX.csv",                 # BENIGN-heavy baseline
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]

df = load_cicids_multi(
    files=FILES_FOR_PHASE_3_5,
    sample_per_file=30000,
    random_state=42
)

df.shape


(90000, 79)

In [8]:
df["Label"].value_counts().head(25)


Label
BENIGN      56236
DDoS        17101
PortScan    16663
Name: count, dtype: int64

## Cleaning pipeline (conservative, reproducible)

The dataset is cleaned conservatively:
- strip whitespace from column names
- replace ±inf with NaN
- remove missing values
- remove exact duplicates

No scaling, encoding, or feature engineering is performed.


In [15]:
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)

before_dropna = df.shape[0]
df.dropna(inplace=True)
after_dropna = df.shape[0]

before_dups = df.shape[0]
df.drop_duplicates(inplace=True)
after_dups = df.shape[0]

print("After cleaning shape:", df.shape)
print(f"Rows removed (missing): {before_dropna - after_dropna}")
print(f"Rows removed (duplicates): {before_dups - after_dups}")


After cleaning shape: (87521, 79)
Rows removed (missing): 69
Rows removed (duplicates): 2410


## SOC Alert Categories (Network Telemetry Only)

Since CICIDS is a **flow telemetry dataset**, we focus on detections that are realistically supported by flow features.

### Detection categories we will design:

1) **Reconnaissance / Scanning**
- Port scans
- High connection attempts
- Abnormal TCP handshake patterns

2) **DoS / DDoS Flood Behavior**
- High packet rate / byte rate anomalies
- Short repetitive flows

3) **Brute Force (Network Proxy)**
- repeated short session attempts
- timing patterns

4) **Suspicious session behavior / infiltration proxy**
- unusual flow durations / ratios

5) **Large outbound transfer (Exfil proxy)**
- unusually large forward bytes / asymmetric ratios

---

### Key rule (SOC correctness)
We will NOT claim:
- authentication anomalies
- OS-level privilege detection

Because CICIDS does not contain OS event logs.


In [9]:
cols = df.columns.tolist()
len(cols), cols[:20]


(79,
 ['Destination Port',
  'Flow Duration',
  'Total Fwd Packets',
  'Total Backward Packets',
  'Total Length of Fwd Packets',
  'Total Length of Bwd Packets',
  'Fwd Packet Length Max',
  'Fwd Packet Length Min',
  'Fwd Packet Length Mean',
  'Fwd Packet Length Std',
  'Bwd Packet Length Max',
  'Bwd Packet Length Min',
  'Bwd Packet Length Mean',
  'Bwd Packet Length Std',
  'Flow Bytes/s',
  'Flow Packets/s',
  'Flow IAT Mean',
  'Flow IAT Std',
  'Flow IAT Max',
  'Flow IAT Min'])

In [10]:
feature_groups = {
    "volume": [c for c in cols if ("Bytes" in c) or ("Pkt" in c) or ("Packet" in c)],
    "duration": [c for c in cols if ("Duration" in c)],
    "iat": [c for c in cols if ("IAT" in c)],
    "tcp_flags": [c for c in cols if ("Flag" in c)],
    "rates": [c for c in cols if ("Rate" in c) or ("/s" in c)],
    "bulk": [c for c in cols if ("Bulk" in c)],
    "header": [c for c in cols if ("Header" in c)],
}

{k: len(v) for k, v in feature_groups.items()}


{'volume': 30,
 'duration': 1,
 'iat': 14,
 'tcp_flags': 12,
 'rates': 6,
 'bulk': 6,
 'header': 3}

## Signal Candidate Mapping Table

We will generate a structured mapping table:

| SOC Alert | Attack Label | Candidate Features | Why It Works | False Positive Risk |

This is what makes this project *architect-track* instead of “random data science”.


In [11]:
signal_map = pd.DataFrame([
    {
        "soc_alert": "Reconnaissance / Port Scan",
        "expected_attack_labels": "PortScan",
        "candidate_features": "Flow Pkts/s, Tot Fwd Pkts, Tot Bwd Pkts, SYN Flag Cnt, Flow IAT Mean",
        "why_it_works": "Scanning produces many small flows and abnormal SYN/flow timing patterns.",
        "main_false_positive_risk": "IT scanning, vulnerability scanners, monitoring tools"
    },
    {
        "soc_alert": "DoS / DDoS Flood Behavior",
        "expected_attack_labels": "DDoS, DoS Hulk, DoS GoldenEye",
        "candidate_features": "Flow Bytes/s, Flow Pkts/s, Packet Length Mean, Flow Duration",
        "why_it_works": "Flooding creates rate spikes and unusual distribution of packet sizes and durations.",
        "main_false_positive_risk": "Legitimate traffic surges, backups, high-traffic services"
    },
    {
        "soc_alert": "Brute Force / Repeated Attempts (Network Proxy)",
        "expected_attack_labels": "FTP-Patator, SSH-Patator",
        "candidate_features": "Flow Duration, Flow IAT Mean, Tot Fwd Pkts, Active Mean",
        "why_it_works": "Repeated access attempts tend to produce repeated short flows with consistent timing structure.",
        "main_false_positive_risk": "Automation scripts, misconfigured clients, load testing"
    },
    {
        "soc_alert": "Web Attack / Suspicious Session Behavior (Proxy)",
        "expected_attack_labels": "Web Attack – Brute Force, Web Attack – XSS",
        "candidate_features": "Flow Duration, Fwd Pkt Len Mean, Flow IAT Std, Bwd Pkt Len Mean",
        "why_it_works": "Web attacks may affect session timing and directional packet size patterns.",
        "main_false_positive_risk": "Normal browsing bursts, content-heavy pages, API bursts"
    },
    {
        "soc_alert": "Large Outbound Transfer (Exfil Proxy)",
        "expected_attack_labels": "Infiltration (proxy), Exfil-like behavior",
        "candidate_features": "TotLen Fwd Pkts, Flow Bytes/s, Down/Up Ratio, Packet Length Std",
        "why_it_works": "High forward payload volume and asymmetry can proxy large outbound transfers.",
        "main_false_positive_risk": "File uploads, sync tools, legitimate bulk data movement"
    },
    {
        "soc_alert": "Protocol / Infrastructure Anomaly (Proxy)",
        "expected_attack_labels": "Various",
        "candidate_features": "SYN Flag Cnt, RST Flag Cnt, ACK Flag Cnt, PSH Flag Cnt, URG Flag Cnt",
        "why_it_works": "Protocol manipulation or unstable sessions distort flag distributions.",
        "main_false_positive_risk": "Retransmissions, packet loss, unstable networks"
    },
])

signal_map


,soc_alert,expected_attack_labels,candidate_features,why_it_works,main_false_positive_risk
0,Reconnaissance / Port Scan,PortScan,"Flow Pkts/s, Tot Fwd Pkts, Tot Bwd Pkts, SYN F...",Scanning produces many small flows and abnorma...,"IT scanning, vulnerability scanners, monitorin..."
1,DoS / DDoS Flood Behavior,"DDoS, DoS Hulk, DoS GoldenEye","Flow Bytes/s, Flow Pkts/s, Packet Length Mean,...",Flooding creates rate spikes and unusual distr...,"Legitimate traffic surges, backups, high-traff..."
2,Brute Force / Repeated Attempts (Network Proxy),"FTP-Patator, SSH-Patator","Flow Duration, Flow IAT Mean, Tot Fwd Pkts, Ac...",Repeated access attempts tend to produce repea...,"Automation scripts, misconfigured clients, loa..."
3,Web Attack / Suspicious Session Behavior (Proxy),"Web Attack – Brute Force, Web Attack – XSS","Flow Duration, Fwd Pkt Len Mean, Flow IAT Std,...",Web attacks may affect session timing and dire...,"Normal browsing bursts, content-heavy pages, A..."
4,Large Outbound Transfer (Exfil Proxy),"Infiltration (proxy), Exfil-like behavior","TotLen Fwd Pkts, Flow Bytes/s, Down/Up Ratio, ...",High forward payload volume and asymmetry can ...,"File uploads, sync tools, legitimate bulk data..."
5,Protocol / Infrastructure Anomaly (Proxy),Various,"SYN Flag Cnt, RST Flag Cnt, ACK Flag Cnt, PSH ...",Protocol manipulation or unstable sessions dis...,"Retransmissions, packet loss, unstable networks"


In [12]:
def parse_features(feature_string: str):
    parts = [x.strip() for x in feature_string.split(",")]
    return [p for p in parts if p]

all_requested = []
for s in signal_map["candidate_features"].tolist():
    all_requested.extend(parse_features(s))

all_requested = sorted(set(all_requested))

availability = pd.DataFrame({
    "feature": all_requested,
    "exists_in_dataset": [f in df.columns for f in all_requested]
})

availability


,feature,exists_in_dataset
0,ACK Flag Cnt,False
1,Active Mean,True
2,Bwd Pkt Len Mean,False
3,Down/Up Ratio,True
4,Flow Bytes/s,True
5,Flow Duration,True
6,Flow IAT Mean,True
7,Flow IAT Std,True
8,Flow Pkts/s,False
9,Fwd Pkt Len Mean,False


In [16]:
label_col = "Label"
normal_data = df[df[label_col] == "BENIGN"]
attack_data = df[df[label_col] != "BENIGN"]

print("Normal rows:", normal_data.shape[0])
print("Attack rows:", attack_data.shape[0])


Normal rows: 55230
Attack rows: 32291


In [17]:
signal_features = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Flow Bytes/s"
]

baseline_quantiles = {}
for f in signal_features:
    if f in df.columns:
        baseline_quantiles[f] = {
            "p95_benign": normal_data[f].quantile(0.95),
            "p99_benign": normal_data[f].quantile(0.99),
            "p999_benign": normal_data[f].quantile(0.999),
        }

baseline_q_df = pd.DataFrame(baseline_quantiles).T
baseline_q_df


,p95_benign,p99_benign,p999_benign
Flow Duration,1.061847e+08,1.177645e+08,1.198062e+08
Total Fwd Packets,2.000000e+01,5.100000e+01,2.654810e+02
Total Backward Packets,1.800000e+01,6.100000e+01,3.697710e+02
Total Length of Fwd Packets,4.411650e+03,1.161300e+04,2.885394e+04
Flow Bytes/s,2.530612e+06,1.233333e+07,1.344608e+08


## Interpretation

### What this tells us
- We now have a SOC-aligned mapping table
- We verified candidate columns exist in the dataset
- This becomes the “design spec” for Notebook 08/09

### Next steps
Notebook 08 will test:
- which signals are noisy in BENIGN traffic
- which ones have high false-positive rates
- what thresholds should be treated as “investigate” vs “ignore”


In [13]:
out_map = OUTPUT_TABLES / "08_signal_candidate_mapping.csv"
out_avail = OUTPUT_TABLES / "08_feature_availability.csv"

signal_map.to_csv(out_map, index=False)
availability.to_csv(out_avail, index=False)

out_map, out_avail


(WindowsPath('D:/Projects/CICIDS-2017/03-security-integration/cicids-2017/phase-3.5-soc-detection/outputs/tables/08_signal_candidate_mapping.csv'),
 WindowsPath('D:/Projects/CICIDS-2017/03-security-integration/cicids-2017/phase-3.5-soc-detection/outputs/tables/08_feature_availability.csv'))

## Notebook 08 Status — Completed ✅

### Completed
- Multi-file CICIDS loading (no merged giant file)
- SOC alert categories formally mapped to CICIDS signal candidates
- Feature existence validated
- Outputs exported

### Next
Notebook 08 — False Positive Stress Test
